# Ablation — No RAG (Reasoning only, zero-shot)

Removes the retrieval module from the full pipeline.
The model receives the trait definition + scoring note + structured reasoning prompt,
but **no retrieved exemplars** (`similar_context` is empty).

Compare against full system (`reasoned_rag_def_oneshot_30f`) to isolate the contribution of RAG.

In [1]:
from pathlib import Path
import sys, os, json, time
from typing import Dict, Optional

import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / "ptd_model").exists():
    project_root = (project_root / ".." / "..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ptd_model.prompts import (
    SYS_PROMPT_REASONED,
    REASONED_RAG_DEF_ONESHOT_PROMPT,
    TRAITS,
    TRAIT_NOTES,
)
from ptd_model.evaluate import evaluate
from utils.gpt_client import gpt_call
from utils.log import log_to_file
from utils.parser import extract_reasoned_full

print("Project root:", project_root)

Project root: F:\std\GR\code\model_x_ocean


## Configuration

In [2]:
test_csv   = str(project_root / "data/split/essays/test.csv")
res_dir    = str(project_root / "result")
log_dir    = str(project_root / "log")

model_name     = "gpt-4o-mini"
max_new_tokens = 512
temperature    = 0.0
prompt_mode    = "ablation_no_rag_reasoned"   # distinct save folder

TRAIT_NAMES = ("Openness", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism")
TRAIT_TO_COLUMN = {
    "Openness":          "pred_cOPN",
    "Conscientiousness": "pred_cCON",
    "Extraversion":      "pred_cEXT",
    "Agreeableness":     "pred_cAGR",
    "Neuroticism":       "pred_cNEU",
}

test_df = pd.read_csv(test_csv)
print(f"Test: {len(test_df)} rows | mode={prompt_mode} | model={model_name}")

Test: 247 rows | mode=ablation_no_rag_reasoned | model=gpt-4o-mini


## Predict — reasoning prompt, no exemplars

`similar_context` is intentionally left empty to ablate RAG.
Everything else (system prompt, XML reasoning chain, trait definitions, scoring notes) is identical to the full system.

In [3]:
run_id       = time.strftime("%Y%m%d-%H%M%S")
safe_model   = model_name.replace(":", "_")
log_filepath = os.path.join(log_dir, safe_model, prompt_mode, f"{run_id}_log.txt")
output_dir   = os.path.join(res_dir, safe_model, prompt_mode, run_id)
os.makedirs(output_dir, exist_ok=True)

reasoning_log_path = os.path.join(output_dir, "reasoning_log.jsonl")

df = test_df.copy()
for col in TRAIT_TO_COLUMN.values():
    df[col] = None

n  = len(df)
t0 = time.time()
print(f"[predict] {n} records | mode={prompt_mode} | model={model_name}")

for idx, row in df.iterrows():
    text = row["text"]

    for trait_name, pred_col in TRAIT_TO_COLUMN.items():
        trait_defs = TRAITS.get(trait_name, {})
        trait_note = TRAIT_NOTES.get(trait_name, "")

        # --- ablation: similar_context is empty ---
        usr_prompt = REASONED_RAG_DEF_ONESHOT_PROMPT.format(
            trait_name=trait_name,
            definition_high=trait_defs.get("high", ""),
            definition_low=trait_defs.get("low", ""),
            top_k=0,
            similar_context="(no retrieved examples)",
            trait_note=trait_note,
        )

        formatted = usr_prompt.replace("<text>", text)
        output = gpt_call(formatted, SYS_PROMPT_REASONED, model_name, max_new_tokens, temperature)
        log_to_file(log_filepath, SYS_PROMPT_REASONED, formatted, output, f"{idx}-{trait_name}")

        parsed = extract_reasoned_full(output.strip())
        df.at[idx, pred_col] = parsed.get("label")

        rec = {
            "record_idx":        int(idx),
            "trait":             trait_name,
            "label":             parsed.get("label"),
            "evidence":          parsed.get("evidence"),
            "facet_check":       parsed.get("facet_check"),
            "example_alignment": parsed.get("example_alignment"),
            "verdict":           parsed.get("verdict"),
        }
        with open(reasoning_log_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    if (idx + 1) % 10 == 0:
        print(f"  [predict] {idx + 1}/{n} done.")

elapsed = time.time() - t0
prediction_csv = os.path.join(output_dir, "predictions.csv")
df.to_csv(prediction_csv, index=False)
print(f"[predict] Finished in {elapsed:.1f}s -> {prediction_csv}")

[predict] 247 records | mode=ablation_no_rag_reasoned | model=gpt-4o-mini
  [predict] 10/247 done.
  [predict] 20/247 done.
  [predict] 30/247 done.
  [predict] 40/247 done.
  [predict] 50/247 done.
  [predict] 60/247 done.
  [predict] 70/247 done.
  [predict] 80/247 done.
  [predict] 90/247 done.
  [predict] 100/247 done.
  [predict] 110/247 done.
  [predict] 120/247 done.
  [predict] 130/247 done.
  [predict] 140/247 done.
  [predict] 150/247 done.
  [predict] 160/247 done.
  [predict] 170/247 done.
  [predict] 180/247 done.
  [predict] 190/247 done.
  [predict] 200/247 done.
  [predict] 210/247 done.
  [predict] 220/247 done.
  [predict] 230/247 done.
  [predict] 240/247 done.
[predict] Finished in 6478.8s -> F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\ablation_no_rag_reasoned\20260531-002623\predictions.csv


## Evaluate

In [ ]:
evaluation = evaluate(
    prediction_csv = prediction_csv,
    model_name     = model_name,
    res_dir        = res_dir,
    run_time       = elapsed,
    prompt_mode    = prompt_mode,
    run_id         = run_id,
)

print("Summary CSV:", evaluation["summary_csv"])
print(f"Failed predictions: {evaluation['fail_count']} / {evaluation['n_records']}")
summary_df = pd.read_csv(evaluation["summary_csv"])
display(summary_df[["trait", "n_samples", "accuracy", "macro_f1", "weighted_f1"]]
        .sort_values("accuracy", ascending=False)
        .reset_index(drop=True))

Loaded predictions from F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\ablation_no_rag_reasoned\20260531-002623\predictions.csv
Saved evaluation summary to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\ablation_no_rag_reasoned\20260531-002623\evaluation_summary.csv
Saved Openness report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\ablation_no_rag_reasoned\20260531-002623\Openness_classification_report.txt
Saved Conscientiousness report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\ablation_no_rag_reasoned\20260531-002623\Conscientiousness_classification_report.txt
Saved Extraversion report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\ablation_no_rag_reasoned\20260531-002623\Extraversion_classification_report.txt
Saved Agreeableness report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\ablation_no_rag_reasoned\20260531-002623\Agreeableness_classification_report.txt
Saved Neuroticism report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\ablation_no_rag_reasone

,trait,n_samples,accuracy,macro_f1,weighted_f1
0,Openness,247,0.562753,0.539942,0.537039
1,Extraversion,247,0.546559,0.463876,0.457909
2,Conscientiousness,247,0.534413,0.451567,0.448978
3,Agreeableness,247,0.530364,0.514241,0.507434
4,Neuroticism,247,0.506073,0.356771,0.355516


: 